# Module 05 Lab — Data Preparation & Feature Engineering
## CLINICPROOF™ Edition — Sensor-Verified Medical Procedure Compliance

**Course:** ITAI 1371 — Introduction to AI  
**Project:** CLINICPROOF™ — Healthcare AI / Human Activity Recognition

---

**Objective:** To learn and apply the most common data preparation and feature engineering techniques **on real healthcare sensor data**. Raw IMU sensor readings from wearable devices are never ready for a machine learning model — they have missing values from sensor dropouts, are recorded on wildly different scales, and don't carry the clinical meaning that a hand-hygiene-verification model actually needs to make safe predictions.

This process — also called preprocessing — is one of the most critical steps in the entire ML workflow. **In healthcare AI, it's also a patient-safety step:** poorly prepared data leads to silently failing models that pass audits but miss real procedure violations.

**In this lab, you will write more of the code.** Read the explanations and then complete the tasks in the code cells.

### 🩺 Why This Matters
The CLINICPROOF abstract estimates that proper procedure compliance — the thing this preprocessed data eventually enables — could prevent **30–50% of hospital-acquired infections** and save **$5.8–16.5M annually per 500-bed hospital**. Every preprocessing decision you make in this lab corresponds to a real clinical failure mode you're either preventing or enabling.


## Part 1: Setup and Initial Look

We will use a **simulated CLINICPROOF sensor dataset** that mirrors the UCI HAR dataset structure (the foundation specified in the CLINICPROOF abstract and PRD). The dataset contains the exact problems we need to solve:

- **Missing values** — sensor dropouts during procedures (heart rate sensor occasionally loses contact)
- **Non-numeric data** — categorical activity labels and shift periods that ML models can't process directly
- **Different scales** — accelerometer values around 10 m/s², gyroscope values around 0.3 rad/s, heart rate around 78 BPM
- **Raw IMU readings without clinical meaning** — `acc_x_mean = 0.5` doesn't tell a nurse anything actionable

Each section below tackles one of these problems.


In [3]:
import pandas as pd
import numpy as np

# Reproducibility — required by the CLINICPROOF PRD (random_state=42 everywhere)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ─── Generate the CLINICPROOF sensor dataset ────────────────────────────
# This simulates what a pilot hospital would send back after a 1-day data collection.
# Real production code would replace this with a load from the UCI HAR dataset:
#   from ucimlrepo import fetch_ucirepo; har = fetch_ucirepo(id=240)

n_samples = 1500
activities = {
    1: 'WALKING', 2: 'WALKING_UPSTAIRS', 3: 'WALKING_DOWNSTAIRS',
    4: 'SITTING', 5: 'STANDING', 6: 'LAYING'
}
activity_ids = np.random.choice(list(activities.keys()), size=n_samples,
                                p=[0.20, 0.10, 0.10, 0.22, 0.12, 0.26])

# Movement factor: how much real motion is happening for each activity
movement_factor = pd.Series(activity_ids).map({
    1: 1.8, 2: 2.4, 3: 2.2, 4: 0.4, 5: 1.2, 6: 0.2
}).values

df = pd.DataFrame({
    'window_id':           range(1, n_samples + 1),
    'subject_id':          np.random.randint(1, 31, size=n_samples),
    'activity_name':       [activities[a] for a in activity_ids],
    'shift_period':        np.random.choice(['day_shift', 'evening_shift', 'night_shift'],
                                            size=n_samples, p=[0.45, 0.35, 0.20]),

    # Accelerometer readings (m/s²) — large numerical range
    'acc_magnitude_mean':  np.random.normal(10.2, 0.6, size=n_samples) + movement_factor * 0.8,
    'acc_magnitude_std':   np.random.gamma(2, 0.5, size=n_samples) * movement_factor,

    # Gyroscope readings (rad/s) — small numerical range
    'gyro_magnitude_mean': np.random.normal(0.3, 0.2, size=n_samples) * movement_factor,
    'gyro_magnitude_std':  np.random.gamma(1.5, 0.1, size=n_samples) * movement_factor,

    # Heart rate (BPM) — physiological range, occasionally drops out
    'heart_rate_bpm':      np.random.normal(78, 12, size=n_samples) + movement_factor * 4,

    # Frequency-domain feature (Beta distribution, range 0–1)
    'fft_energy_high':     np.random.beta(2, 5, size=n_samples) + (movement_factor > 1) * 0.2,

    # Window duration in seconds (target: 2.56s per UCI HAR spec)
    'window_duration_s':   np.random.normal(2.56, 0.04, size=n_samples),
})

# Inject realistic missing values (sensor dropouts) — 8% of heart rate readings
hr_drop_idx = np.random.choice(df.index, size=int(0.08 * len(df)), replace=False)
df.loc[hr_drop_idx, 'heart_rate_bpm'] = np.nan

# Inject a few missing fft values too (sensor calibration glitches)
fft_drop_idx = np.random.choice(df.index, size=int(0.03 * len(df)), replace=False)
df.loc[fft_drop_idx, 'fft_energy_high'] = np.nan

print(f"✅ CLINICPROOF dataset loaded: {df.shape[0]:,} sensor windows, {df.shape[1]} columns")
print()

# Let's look at the missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())
print()
print("--- Data Types ---")
print(df.dtypes)
print()
print("--- First 5 rows ---")
df.head()


✅ CLINICPROOF dataset loaded: 1,500 sensor windows, 11 columns

--- Missing Values Before Cleaning ---
window_id                0
subject_id               0
activity_name            0
shift_period             0
acc_magnitude_mean       0
acc_magnitude_std        0
gyro_magnitude_mean      0
gyro_magnitude_std       0
heart_rate_bpm         120
fft_energy_high         45
window_duration_s        0
dtype: int64

--- Data Types ---
window_id                int64
subject_id               int32
activity_name              str
shift_period               str
acc_magnitude_mean     float64
acc_magnitude_std      float64
gyro_magnitude_mean    float64
gyro_magnitude_std     float64
heart_rate_bpm         float64
fft_energy_high        float64
window_duration_s      float64
dtype: object

--- First 5 rows ---


,window_id,subject_id,activity_name,shift_period,acc_magnitude_mean,acc_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,heart_rate_bpm,fft_energy_high,window_duration_s
0,1,16,WALKING_DOWNSTAIRS,day_shift,12.003222,1.117315,0.091985,0.067154,84.256613,0.289175,2.528515
1,2,2,LAYING,day_shift,10.232675,0.114192,0.043721,0.016934,85.443529,0.298528,2.579233
2,3,6,STANDING,day_shift,10.588849,0.496992,0.363361,0.338252,87.843718,0.461426,2.527043
3,4,28,SITTING,day_shift,10.566488,0.695846,0.165103,0.075677,86.701614,0.175444,2.563151
4,5,21,WALKING,day_shift,11.794652,0.639020,0.327098,0.361827,NaN,0.760391,2.578479


## Part 2: Handling Missing Values (Imputation)

**Concept:** Most machine learning models cannot handle missing values (`NaN`). We must deal with them.

In a healthcare context, missing values usually come from **sensor dropouts** — the heart rate monitor losing contact with the wrist for a few seconds, or a Bluetooth Low Energy (BLE) beacon failing to ping. Dropping the rows is an option, but you lose data — and in healthcare, **the missing rows might be exactly the safety-critical ones** (e.g., the heart rate sensor often fails during fast hand movements, which is exactly when a clinician is rushing through hand hygiene).

A better way is **imputation**, which means filling in the missing values with a calculated guess.

### Common imputation strategies:
- **Mean:** Fill with the average value. Good for normally distributed data (e.g., `acc_magnitude_mean`).
- **Median:** Fill with the middle value. Better for skewed data or data with outliers (e.g., `heart_rate_bpm` — a few stressed clinicians might have very high readings).
- **Mode:** Fill with the most frequent value. Used for categorical data (e.g., if `activity_name` were ever missing).
- **Forward-fill / Group-aware imputation:** Use surrounding readings — particularly useful for time-series sensor data.

### 🩺 The CLINICPROOF Decision
For `heart_rate_bpm`, we use the **median** because heart rate distributions are often skewed in clinical environments — a small number of clinicians experiencing high stress can pull the mean upward and make the imputed value misleading.


### Task 1: Impute the 'heart_rate_bpm' Column

The `heart_rate_bpm` column is missing many values from sensor dropouts. Since heart rate can be skewed (e.g., by stressed or rushed clinicians), using the **median** is a robust choice.

**Your Task:** Calculate the median of the `heart_rate_bpm` column and use the `.fillna()` method to replace the missing values.


In [ ]:
# --- ENTER YOUR CODE HERE ---

# 1. Calculate the median of the 'heart_rate_bpm' column
# median_hr = ...

# 2. Fill the missing values in 'heart_rate_bpm' with the median
#    NOTE: the modern pandas pattern is to reassign, not use inplace=True
# df['heart_rate_bpm'] = df['heart_rate_bpm'].fillna(...)

# 3. Also impute 'fft_energy_high' — use mean here (this feature is symmetric, range 0-1)
# df['fft_energy_high'] = df['fft_energy_high'].fillna(...)

# 4. Verify that there are no more missing values
# print("Missing values after imputation:")
# print(df.isnull().sum())


## Part 3: Encoding Categorical Features

**Concept:** Machine learning models are mathematical, so they need numbers, not text. We need to convert categorical columns (like `activity_name` and `shift_period`) into a numerical format.

In CLINICPROOF, almost every clinically-meaningful column is categorical:
- `activity_name` — the **target variable** (the thing we're predicting)
- `shift_period` — when the data was collected (day/evening/night)
- `motion_category` — the engineered clinician-readable label (we'll create this in Part 5)
- `data_quality_flag` — whether the sensor data is trustworthy

There are two common encoding strategies, and **picking the wrong one will silently destroy model performance**:

### Strategy 1: Label Encoding (for ordered categories or target variables)
Maps each category to a single integer: `day_shift → 0`, `evening_shift → 1`, `night_shift → 2`.

✅ Use for: ordinal features (categories with a real order, like Low/Medium/High)  
✅ Use for: target variables in classification (the model treats them as just labels)  
❌ Don't use for: nominal categorical features (e.g., `activity_name`) — you'll teach the model that `WALKING (1) + STANDING (5) = LAYING (6)`, which is nonsense.

### Strategy 2: One-Hot Encoding (for nominal categories used as features)
Takes a column with `N` categories and turns it into `N` new columns, each with a `1` or `0`.

For example, the `shift_period` column (`day_shift`, `evening_shift`, `night_shift`) becomes three new columns: `shift_day_shift`, `shift_evening_shift`, `shift_night_shift`.

Pandas has a convenient function called `pd.get_dummies()` that does this for us.

### 🩺 The CLINICPROOF Decision
- `activity_name` (target) → **Label Encoding** (each activity gets an integer ID 0–5)
- `shift_period` (feature) → **One-Hot Encoding** (no inherent order; day vs. night should be independent dimensions)


### Task 2: Encode the Categorical Columns

**Your Task:** 
1. Use `pd.get_dummies()` to one-hot encode the `shift_period` column (it's a feature with no inherent order).
2. Convert the `activity_name` column into a numeric `activity_id` (label encoding) — this is our **target variable** for the eventual classification model.


In [2]:
# 1. One-hot encode the 'shift_period' column
#    drop_first=True keeps k-1 columns to avoid perfect multicollinearity.
encoded_df = pd.get_dummies(df, columns=['shift_period'], drop_first=True)

# 2. Label-encode the 'activity_name' column into a numeric target.
activity_to_id = {
    'LAYING': 0,
    'SITTING': 1,
    'STANDING': 2,
    'WALKING': 3,
    'WALKING_UPSTAIRS': 4,
    'WALKING_DOWNSTAIRS': 5
}
encoded_df['activity_id'] = encoded_df['activity_name'].map(activity_to_id)

# 3. Drop the original text target column.
encoded_df = encoded_df.drop(columns=['activity_name'])

# 4. Display a quick verification view.
print(encoded_df.head())
print(f"\nFinal column count: {encoded_df.shape[1]}")
print(f"Unique encoded labels: {sorted(encoded_df['activity_id'].dropna().unique().tolist())}")

   window_id  subject_id  acc_magnitude_mean  acc_magnitude_std  \
0          1          16           12.003222           1.117315   
1          2           2           10.232675           0.114192   
2          3           6           10.588849           0.496992   
3          4          28           10.566488           0.695846   
4          5          21           11.794652           0.639020   

   gyro_magnitude_mean  gyro_magnitude_std  heart_rate_bpm  fft_energy_high  \
0             0.091985            0.067154       84.256613         0.289175   
1             0.043721            0.016934       85.443529         0.298528   
2             0.363361            0.338252       87.843718         0.461426   
3             0.165103            0.075677       86.701614         0.175444   
4             0.327098            0.361827             NaN         0.760391   

   window_duration_s  shift_period_evening_shift  shift_period_night_shift  \
0           2.528515                       F

## Part 4: Feature Scaling

**Concept:** Many models are sensitive to the scale of the features. Look at the CLINICPROOF sensor data we have:

| Feature | Typical Range | Units |
|---|---|---|
| `acc_magnitude_mean` | 8 – 14 | m/s² |
| `gyro_magnitude_mean` | 0.0 – 1.5 | rad/s |
| `heart_rate_bpm` | 50 – 130 | beats per minute |
| `fft_energy_high` | 0.0 – 1.0 | normalized |
| `window_duration_s` | 2.5 – 2.6 | seconds |

A KNN or SVM model looking at these features will incorrectly believe that `heart_rate_bpm` is the most important feature simply because its values are 50–100× larger than the gyroscope readings. **In a clinical setting, that's a direct path to a misclassified hand-hygiene event.**

**Feature Scaling** solves this by putting all features on a similar scale. A common method is **Standardization** (`StandardScaler` in scikit-learn), which rescales the data to have a mean of 0 and a standard deviation of 1.

### ⚠️ Two critical rules

**Rule 1: Don't scale the target variable.** We never scale `activity_id` — it's the thing we're predicting, not a feature.

**Rule 2: Don't scale categorical features.** The newly-created one-hot columns (like `shift_evening_shift`) are already in 0/1 form. Scaling them would distort their meaning.

**Important:** In a real production pipeline, you `.fit()` the scaler on training data only, then `.transform()` both the training and test data. Otherwise you leak information from the test set into the training process. We'll cover that in Module 6, but for this lab we're scaling the whole dataset to keep things simple.


### Task 3: Scale the Numeric Sensor Features

**Your Task:** Use `StandardScaler` from `sklearn.preprocessing` to scale the 5 numeric sensor features.


In [ ]:
from sklearn.preprocessing import StandardScaler

# --- ENTER YOUR CODE HERE ---

# 1. Create an instance of the StandardScaler
# scaler = ...

# 2. Select the columns to scale
#    NOTE: Do NOT include 'activity_id' (target), one-hot columns, or ID columns.
# columns_to_scale = ['acc_magnitude_mean', 'acc_magnitude_std',
#                     'gyro_magnitude_mean', 'gyro_magnitude_std',
#                     'heart_rate_bpm', 'fft_energy_high', 'window_duration_s']

# 3. Fit the scaler to the data and transform it.
#    Note: We are using the `encoded_df` from the previous step.
# encoded_df[columns_to_scale] = scaler.fit_transform(encoded_df[columns_to_scale])

# 4. Display the first few rows to see the scaled data
#    After scaling, each of these columns should have mean ≈ 0 and std ≈ 1.
# print(encoded_df[columns_to_scale].describe().round(3))
# print()
# print(encoded_df.head())


## Part 5: Feature Engineering — The CLINICPROOF Edge

**Concept:** Imputation, encoding, and scaling are *cleaning* operations — they make existing features model-ready. **Feature engineering is creating new features from existing ones** to give the model information it couldn't easily learn on its own.

This is where CLINICPROOF goes from "generic activity recognition" to "clinical compliance verification." The CLINICPROOF PRD (Section 5) specifies exactly four categories of clinically-meaningful engineered features:

1. **Gesture duration** — how long was a hand-hygiene motion? (must be ≥ 20 seconds per WHO guidelines)
2. **Motion intensity** — was this a routine movement or an urgent rushed one?
3. **Spatial patterns** — room-to-room transitions
4. **Frequency analysis** — repetitive rubbing motions (hand washing has a characteristic frequency signature)

A raw `acc_magnitude_mean = 11.4` is meaningless to a clinician. An engineered `motion_category = 'active'` is something a charge nurse can immediately understand and audit.

### 🩺 Why This Matters Beyond Accuracy
Engineered features are also **interpretable features**. When CLINICPROOF flags a non-compliance event, the system needs to explain *why* — "the model predicted non-compliance" is useless to a hospital. "Motion intensity was 'stationary' during a window labeled STANDING-near-sink, indicating no actual hand-washing motion" is actionable.

This interpretability is also what makes the system defensible in court when an actual lawsuit happens — and per the abstract, healthcare malpractice is a $3–4B/year problem.


### Task 4: Engineer Three Clinical-Meaning Features

**Your Task:** Create three new features that a clinician would actually find useful when reviewing a flagged compliance event.

1. **`total_motion_intensity`** — A single combined score of how much physical motion is happening in this window. Combine `acc_magnitude_mean` and `gyro_magnitude_mean` with a weighted sum (try 60/40).

2. **`motion_category`** — Categorize the intensity into clinician-readable bins: `stationary`, `routine`, `active`, or `urgent`. Use these thresholds (calibrated from the actual data distribution): `< 6.5` → stationary, `< 7.0` → routine, `< 7.5` → active, `≥ 7.5` → urgent.

3. **`is_repetitive_motion`** — A binary flag (0 or 1) indicating whether this window contains the high-frequency repetitive motion characteristic of hand-washing or scrubbing. Use `fft_energy_high > 0.5` as your threshold.

> 💡 **Important:** Do this BEFORE scaling. Feature engineering uses the original (unscaled) values because the thresholds (`< 6.5`, `< 7.0`, etc.) are chosen to mean something in the original units. If you do it after scaling, your thresholds become meaningless.


In [5]:
# Note: For this task, work on the imputed-but-unscaled DataFrame.
# We'll re-create one for clarity:
fe_df = df.copy()
fe_df['heart_rate_bpm'] = fe_df['heart_rate_bpm'].fillna(fe_df['heart_rate_bpm'].median())
fe_df['fft_energy_high'] = fe_df['fft_energy_high'].fillna(fe_df['fft_energy_high'].mean())

# 1. Create 'total_motion_intensity' as a weighted sum (60/40)
fe_df['total_motion_intensity'] = (
    fe_df['acc_magnitude_mean'] * 0.6
    + fe_df['gyro_magnitude_mean'] * 0.4
)

# 2. Create 'motion_category' using calibrated thresholds
def categorize_motion(x):
    if x < 6.5:
        return 'stationary'   # sitting, charting, laying
    elif x < 7.0:
        return 'routine'      # standing, slow walking
    elif x < 7.5:
        return 'active'       # bedside care, walking
    else:
        return 'urgent'       # emergency, stairs

fe_df['motion_category'] = fe_df['total_motion_intensity'].apply(categorize_motion)

# 3. Create 'is_repetitive_motion' as a binary flag
fe_df['is_repetitive_motion'] = (fe_df['fft_energy_high'] > 0.5).astype(int)

# 4. Verify by checking the cross-tab and quick summaries
print("Cross-tab of motion_category vs activity_name:")
print(pd.crosstab(fe_df['activity_name'], fe_df['motion_category']))
print()
print("Motion category distribution:")
print(fe_df['motion_category'].value_counts())
print()
print("Repetitive motion rate:")
print(fe_df['is_repetitive_motion'].value_counts(normalize=True).rename('proportion').round(3))

# Optional preview of engineered columns
fe_df[['acc_magnitude_mean', 'gyro_magnitude_mean', 'total_motion_intensity',
       'motion_category', 'fft_energy_high', 'is_repetitive_motion']].head()

Cross-tab of motion_category vs activity_name:
motion_category     active  routine  stationary  urgent
activity_name                                          
LAYING                   4       80         309       0
SITTING                 12      100         197       0
STANDING                58       81          43       7
WALKING                142       97          17      67
WALKING_DOWNSTAIRS      49       18           2      74
WALKING_UPSTAIRS        52       14           1      76

Motion category distribution:
motion_category
stationary    569
routine       390
active        317
urgent        224
Name: count, dtype: int64

Repetitive motion rate:
is_repetitive_motion
0    0.729
1    0.271
Name: proportion, dtype: float64


,acc_magnitude_mean,gyro_magnitude_mean,total_motion_intensity,motion_category,fft_energy_high,is_repetitive_motion
0,12.003222,0.091985,7.238727,active,0.289175,0
1,10.232675,0.043721,6.157093,stationary,0.298528,0
2,10.588849,0.363361,6.498654,stationary,0.461426,0
3,10.566488,0.165103,6.405934,stationary,0.175444,0
4,11.794652,0.327098,7.207630,active,0.760391,1


## Part 6: Putting It All Together — The Production CLINICPROOF Pipeline

In a real production system, all four steps are chained into a single reproducible pipeline. Below is what the full preprocessing flow looks like end-to-end. **You don't need to write this — just read it carefully and make sure you understand every step.**

The order matters:
1. **Imputation first** — fill missing values before doing anything else
2. **Feature engineering second** — create new features from clean original-scale values
3. **Encoding third** — convert text categories to numbers
4. **Scaling last** — only scale numeric features, never targets or one-hot columns

This ordering is what scikit-learn's `Pipeline` and `ColumnTransformer` enforce automatically — and what your Module 6 lab will introduce.


In [6]:
# Production-ready preprocessing pipeline (read carefully — don't edit)
# This is what your CLINICPROOF model will receive as input.

from sklearn.preprocessing import StandardScaler

prod_df = df.copy()

# ─── Step 1: Imputation ────────────────────────────────────────────
prod_df['heart_rate_bpm'] = prod_df['heart_rate_bpm'].fillna(prod_df['heart_rate_bpm'].median())
prod_df['fft_energy_high'] = prod_df['fft_energy_high'].fillna(prod_df['fft_energy_high'].mean())

# ─── Step 2: Feature Engineering ───────────────────────────────────
prod_df['total_motion_intensity'] = (
    prod_df['acc_magnitude_mean'] * 0.6
    + prod_df['gyro_magnitude_mean'] * 0.4
)

def categorize_motion(x):
    if x < 6.5:   return 'stationary'
    elif x < 7.0: return 'routine'
    elif x < 7.5: return 'active'
    else:         return 'urgent'
prod_df['motion_category'] = prod_df['total_motion_intensity'].apply(categorize_motion)
prod_df['is_repetitive_motion'] = (prod_df['fft_energy_high'] > 0.5).astype(int)

# ─── Step 3: Encoding ──────────────────────────────────────────────
# One-hot encode nominal categorical features
prod_df = pd.get_dummies(prod_df,
                         columns=['shift_period', 'motion_category'],
                         drop_first=True)

# Label encode the target
activity_to_id = {
    'LAYING': 0, 'SITTING': 1, 'STANDING': 2,
    'WALKING': 3, 'WALKING_UPSTAIRS': 4, 'WALKING_DOWNSTAIRS': 5
}
prod_df['activity_id'] = prod_df['activity_name'].map(activity_to_id)
prod_df = prod_df.drop(columns=['activity_name'])

# ─── Step 4: Scaling ───────────────────────────────────────────────
numeric_features = [
    'acc_magnitude_mean', 'acc_magnitude_std',
    'gyro_magnitude_mean', 'gyro_magnitude_std',
    'heart_rate_bpm', 'fft_energy_high',
    'window_duration_s', 'total_motion_intensity'
]
scaler = StandardScaler()
prod_df[numeric_features] = scaler.fit_transform(prod_df[numeric_features])

# ─── Verify the result ─────────────────────────────────────────────
print(f"✅ Production CLINICPROOF feature matrix is ready")
print(f"   Shape: {prod_df.shape[0]:,} rows × {prod_df.shape[1]} columns")
print(f"   Target column: activity_id")
print(f"   Missing values remaining: {prod_df.isnull().sum().sum()}")
print(f"\n--- First 5 rows of production-ready data ---")
prod_df.head()


✅ Production CLINICPROOF feature matrix is ready
   Shape: 1,500 rows × 17 columns
   Target column: activity_id
   Missing values remaining: 0

--- First 5 rows of production-ready data ---


,window_id,subject_id,acc_magnitude_mean,acc_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,heart_rate_bpm,fft_energy_high,window_duration_s,total_motion_intensity,is_repetitive_motion,shift_period_evening_shift,shift_period_night_shift,motion_category_routine,motion_category_stationary,motion_category_urgent,activity_id
0,1,16,1.039787,0.019222,-0.641956,-0.472368,0.145963,-0.559724,-0.807599,0.738757,0,False,False,False,False,False,5
1,2,2,-0.918934,-0.764308,-0.769484,-0.729056,0.246142,-0.509738,0.452720,-0.973199,0,False,False,False,True,False,0
2,3,6,-0.524905,-0.465306,0.075099,0.913288,0.448723,0.360859,-0.844172,-0.432594,0,False,False,False,True,False,2
3,4,28,-0.549642,-0.309983,-0.448757,-0.428807,0.352327,-1.167551,0.053087,-0.579346,0,False,False,False,True,False,1
4,5,21,0.809050,-0.354369,-0.020719,1.033789,0.004069,1.958653,0.433999,0.689538,1,False,False,False,False,False,3


## 📝 Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1. **Why is it often better to impute the `heart_rate_bpm` column with the median instead of the mean?**  
The median is more robust to extreme outliers. In a code-blue or high-stress event, a few very high heart-rate values (for example near 180 BPM) can pull the mean upward and overestimate typical heart rate. The median stays closer to the center of the normal distribution, so imputed values are more stable and clinically realistic.

2. **Explain in your own words what One-Hot Encoding does. Why did we use One-Hot Encoding for `shift_period` but Label Encoding for `activity_name`?**  
One-Hot Encoding converts one categorical column into multiple binary columns, where each column represents one category and has 1 if that row belongs to it, otherwise 0. We used One-Hot for `shift_period` because day/evening/night are nominal categories with no numeric order. We used Label Encoding for `activity_name` because it is the target label for classification, and the model needs class IDs to represent output classes.

3. **Would you need to apply Feature Scaling to a Decision Tree or Random Forest model trained on this CLINICPROOF data?** Why or why not?  
Usually no. Decision Trees and Random Forests split using thresholds on individual features, so they are mostly scale-invariant. A split like `heart_rate_bpm > 90` works the same whether another feature ranges from 0 to 1 or 0 to 1000. Scaling is much more important for distance-based or margin-based models like KNN, SVM, and Logistic Regression.

4. **In Part 5, you engineered three clinical-meaning features.** Which of the three would be most useful to a charge nurse reviewing a flagged non-compliance event, and why? What additional engineered feature would you create if you had more time?
The most useful is `motion_category` because it is immediately interpretable (`stationary`, `routine`, `active`, `urgent`) and supports fast human review without reading raw sensor values. If I had more time, I would add a **sustained-motion-duration** feature (for example, consecutive windows above an intensity threshold) to better capture whether a required procedure was performed long enough to meet protocol.

5. **Order matters in this pipeline.** What would go wrong if you scaled the data BEFORE doing feature engineering? Use one of the engineered features (`motion_category`) to justify your answer.
If scaling happens first, threshold-based engineering loses its real-world meaning. For `motion_category`, thresholds like `< 6.5`, `< 7.0`, and `< 7.5` are calibrated in original sensor units; after scaling, values are standardized around 0 with unit variance, so those thresholds no longer represent clinical intensity. The result would be incorrect category assignments and misleading compliance flags.

---

### 🎯 What's Next
With your CLINICPROOF data now fully preprocessed and feature-engineered, you're ready for **Module 6: Building & Evaluating ML Models**, where you'll feed this `prod_df` into the four PRD-specified classifiers (Logistic Regression, KNN, Linear SVM, Random Forest) and evaluate them against the 90% clinical safety threshold.